# PANICLE Tutorial: End-to-End GWAS on Bundled Demo Data

This notebook walks through a complete GWAS with **PANICLE 0.4+** using the
small demo dataset shipped under `examples/` (sorghum diversity-panel IDs with
plant height and flowering time).

**What you will do**
1. Load and align phenotype/genotype data with `GWASPipeline`
2. Compute PCs (default LOCO MLM does **not** need a precomputed global kinship)
3. Run GLM, LOCO MLM, FarmCPU, and BLINK
4. Inspect results CSVs and simple diagnostic plots
5. Optionally call a lower-level LOCO MLM API

**CLI equivalent** (after `pip install panicle`):
```bash
panicle-gwas \
  --phenotype examples/example_phenotypes.csv \
  --genotype examples/example_genotypes.vcf.gz \
  --traits PlantHeight \
  --methods GLM,MLM,FarmCPU,BLINK \
  --mlm-mode loco \
  --n-pcs 3 \
  --outputdir ./tutorial_results
```



## Setup



In [ ]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path

import matplotlib

# Headless-safe backend for automated execution
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import panicle
from panicle.pipelines.gwas import GWASPipeline
from panicle.association.mlm_loco import PANICLE_MLM_LOCO
from panicle.matrix.kinship_loco import PANICLE_K_VanRaden_LOCO
from panicle.utils.data_types import GenotypeMatrix

warnings.filterwarnings("ignore", category=UserWarning)

print(f"PANICLE {panicle.__version__}")

# Resolve paths whether the notebook runs from repo root or examples/
HERE = Path.cwd().resolve()
if (HERE / "example_phenotypes.csv").exists():
    DATA = HERE
    OUT = HERE / "tutorial_results"
elif (HERE / "examples" / "example_phenotypes.csv").exists():
    DATA = HERE / "examples"
    OUT = HERE / "examples" / "tutorial_results"
else:
    raise FileNotFoundError(
        "Could not find example_phenotypes.csv. "
        "Run from the repo root or the examples/ directory."
    )

PHENOTYPE_FILE = DATA / "example_phenotypes.csv"
GENOTYPE_FILE = DATA / "example_genotypes.vcf.gz"
OUTPUT_DIR = OUT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Phenotype: {PHENOTYPE_FILE}")
print(f"Genotype:  {GENOTYPE_FILE}")
print(f"Output:    {OUTPUT_DIR}")



## 1. Inspect phenotypes

The demo phenotype file has an `ID` column plus numeric traits
`PlantHeight` and `DaysToFlower`.



In [ ]:
pheno_df = pd.read_csv(PHENOTYPE_FILE)
print(f"Samples: {len(pheno_df)}")
print(f"Columns: {list(pheno_df.columns)}")
print(pheno_df.describe().round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, trait in zip(axes, ["PlantHeight", "DaysToFlower"]):
    pheno_df[trait].hist(bins=30, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(trait)
    ax.set_xlabel(trait)
    ax.set_ylabel("Count")
fig.suptitle("Trait distributions")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "trait_distributions.png", dpi=120)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'trait_distributions.png'}")



## 2. High-level pipeline: load, align, structure

- VCF genotypes carry their own marker map (no separate `--map` file).
- With a map present, default `mlm_mode='loco'` builds leave-one-chromosome-out
  kinship during analysis. We only compute PCs here.
- Global kinship (`calculate_kinship=True`) is only required for
  `mlm_mode='global'` or map-less MLM.



In [ ]:
pipeline = GWASPipeline(output_dir=str(OUTPUT_DIR))

t0 = time.time()
pipeline.load_data(
    phenotype_file=str(PHENOTYPE_FILE),
    genotype_file=str(GENOTYPE_FILE),
    trait_columns=["PlantHeight"],  # start with one trait
    loader_kwargs={
        "drop_monomorphic": True,
        "min_maf": 0.0,
    },
)
print(f"Load time: {time.time() - t0:.2f}s")

pipeline.align_samples()
print(
    f"Aligned: {pipeline.genotype_matrix.n_individuals} samples × "
    f"{pipeline.genotype_matrix.n_markers} markers"
)
print(f"Map available: {pipeline.geno_map is not None}")

t0 = time.time()
pipeline.compute_population_structure(
    n_pcs=3,
    calculate_kinship=False,  # LOCO MLM does not need global K
)
print(f"PCA time: {time.time() - t0:.2f}s")
print(f"PCs shape: {pipeline.pcs.shape}")



In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(pipeline.pcs[:, 0], pipeline.pcs[:, 1], alpha=0.45, s=12)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Population structure (PC1 vs PC2)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "pcs.png", dpi=120)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'pcs.png'}")



## 3. Run GWAS (GLM + LOCO MLM + FarmCPU + BLINK)

`mlm_mode='loco'` is the default. FarmCPU pipeline thresholds use library
defaults (`p_threshold=None` → early-stop `0.01/n_tests`).



In [ ]:
t0 = time.time()
pipeline.run_analysis(
    traits=["PlantHeight"],
    methods=["GLM", "MLM", "FarmCPU", "BLINK"],
    mlm_mode="loco",
    max_iterations=5,
    ncpus=1,
    parallel_mode="off",
    min_mac=5,  # demo panel is small; production often uses 10
    outputs=[
        "all_marker_pvalues",
        "significant_marker_pvalues",
        "manhattan",
        "qq",
    ],
)
print(f"Analysis time: {time.time() - t0:.2f}s")



## 4. Inspect results



In [ ]:
results_path = OUTPUT_DIR / "GWAS_PlantHeight_all_results.csv"
results = pd.read_csv(results_path)
print(f"Results: {results.shape[0]} markers × {results.shape[1]} columns")
print("Columns:", list(results.columns)[:12], "...")

# Prefer pipeline threshold from summary if present
summary_path = OUTPUT_DIR / "GWAS_summary_by_traits_methods.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print("\nSummary:")
    print(summary.to_string(index=False))

n_markers = len(results)
bonferroni = 0.05 / max(n_markers, 1)
print(f"\nBonferroni (0.05/M): {bonferroni:.3e}")

method_cols = [c for c in results.columns if c.endswith("_P")]
print("Significant hits (Bonferroni):")
for col in method_cols:
    n_sig = int((results[col] < bonferroni).sum())
    print(f"  {col}: {n_sig}")

top = results.nsmallest(10, "MLM_P")[
    [c for c in ["MARKER", "SNP", "CHROM", "POS", "MAF", "GLM_P", "MLM_P", "FarmCPU_P", "BLINK_P"] if c in results.columns]
]
print("\nTop 10 markers by MLM p-value:")
print(top.to_string(index=False))



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

def scatter_logp(ax, xcol, ycol, title):
    if xcol not in results.columns or ycol not in results.columns:
        ax.set_visible(False)
        return
    x = -np.log10(results[xcol].clip(lower=1e-300))
    y = -np.log10(results[ycol].clip(lower=1e-300))
    ax.scatter(x, y, s=4, alpha=0.25)
    lim = max(float(x.max()), float(y.max()), 1.0)
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    ax.set_xlabel(f"-log10({xcol})")
    ax.set_ylabel(f"-log10({ycol})")
    ax.set_title(title)

scatter_logp(axes[0], "GLM_P", "MLM_P", "GLM vs MLM")
scatter_logp(axes[1], "GLM_P", "FarmCPU_P", "GLM vs FarmCPU")
scatter_logp(axes[2], "MLM_P", "BLINK_P", "MLM vs BLINK")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "method_p_comparison.png", dpi=120)
plt.close(fig)
print(f"Saved {OUTPUT_DIR / 'method_p_comparison.png'}")



In [ ]:
def plot_manhattan(df: pd.DataFrame, p_col: str, out_path: Path) -> None:
    if p_col not in df.columns:
        print(f"Skip Manhattan: missing {p_col}")
        return
    work = df[["CHROM", "POS", p_col]].copy()
    work = work[np.isfinite(work[p_col])]
    work["CHROM"] = work["CHROM"].astype(str)
    chroms = sorted(work["CHROM"].unique(), key=lambda c: (len(c), c))
    offsets = {}
    cursor = 0
    xs, ys, colors = [], [], []
    for i, chrom in enumerate(chroms):
        sub = work[work["CHROM"] == chrom]
        offsets[chrom] = cursor
        xs.append(sub["POS"].to_numpy() + cursor)
        ys.append(-np.log10(sub[p_col].clip(lower=1e-300)))
        colors.append(np.full(len(sub), i % 2))
        cursor += int(sub["POS"].max()) + 1_000_000
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.scatter(
        np.concatenate(xs),
        np.concatenate(ys),
        c=np.concatenate(colors),
        cmap="coolwarm",
        s=6,
        alpha=0.6,
    )
    thr = -np.log10(0.05 / max(len(work), 1))
    ax.axhline(thr, color="red", ls="--", lw=1, label="Bonferroni 0.05/M")
    ax.set_xlabel("Genome position (concatenated chromosomes)")
    ax.set_ylabel(f"-log10({p_col})")
    ax.set_title(f"Manhattan — {p_col}")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)
    print(f"Saved {out_path}")

plot_manhattan(results, "MLM_P", OUTPUT_DIR / "custom_manhattan_MLM.png")



## 5. Low-level LOCO MLM (optional)

For programmatic control, call `PANICLE_MLM_LOCO` with a precomputed LOCO kinship.
High-level APIs no longer accept a user-supplied global `K`.



In [ ]:
# Use the already-aligned pipeline genotype/map/PCs
geno = pipeline.genotype_matrix
gmap = pipeline.geno_map.to_dataframe() if hasattr(pipeline.geno_map, "to_dataframe") else pipeline.geno_map.data
y = pipeline.phenotype_df["PlantHeight"].to_numpy(dtype=float)
mask = np.isfinite(y)
y = y[mask]
geno_sub = geno.subset_individuals(np.where(mask)[0], materialize=True)
pcs_sub = pipeline.pcs[mask]

loco = PANICLE_K_VanRaden_LOCO(geno_sub, gmap, maxLine=1024, cpu=1, verbose=False)
phe = np.column_stack([np.arange(y.size), y])
loco_res = PANICLE_MLM_LOCO(
    phe=phe,
    geno=geno_sub,
    map_data=gmap,
    loco_kinship=loco,
    CV=pcs_sub,
    maxLine=1024,
    cpu=1,
    lrt_refinement=False,
    verbose=False,
)
print(
    f"Low-level LOCO MLM: {loco_res.pvalues.size} markers; "
    f"min p = {np.nanmin(loco_res.pvalues):.3e}"
)



## 6. Summary

- Prefer **`panicle-gwas`** / `GWASPipeline` for routine analyses.
- Default **MLM is LOCO** when a map is available; do not precompute global kinship unless you set `mlm_mode='global'`.
- Pipeline outputs include full and significant CSVs, Manhattan/QQ plots, and a run summary table.
- For multi-trait eQTL-style speedups, see `eqtl_multitrait_acceleration_tutorial.ipynb`.



In [ ]:
print("Files in output directory:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name:55s} {p.stat().st_size:8d} bytes")
print("\nTutorial complete.")

